# 03 — Dual Retrieval & LangGraph Pipeline

**Phase 3 deliverable.** Demonstrates:
1. Tiered Cypher retrieval (T1–T4) with token budget enforcement
2. ChromaDB vector retrieval for tonal anchors
3. Full LangGraph `StoryPipeline` — retrieve → assemble → guard → generate → extract
4. Baseline mode comparison (rolling text summary vs. graph RAG)
5. Quantitative metrics: token usage, retrieval coverage, generation quality

**Prerequisites:** Run `01_schema_and_ingest.ipynb` first to populate Neo4j.

In [ ]:
import os, sys, logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

logging.basicConfig(level=logging.INFO, format="%(name)s | %(levelname)s | %(message)s")
logger = logging.getLogger("nb03")

print(f"Python: {sys.version}")
print(f"Neo4j URI: {os.getenv('NEO4J_URI')}")
print(f"Embedding model: {os.getenv('EMBEDDING_MODEL')}")

## 1. Initialize Components

In [ ]:
from langchain_anthropic import ChatAnthropic
from src.graph_client import GraphClient
from src.extraction import ExtractionPipeline
from src.retrieval import CypherRetriever, VectorRetriever, ContextAssembler, count_tokens
from src.pipeline import StoryPipeline
from src.schema import SessionState

gc = GraphClient(
    uri=os.getenv("NEO4J_URI"),
    user=os.getenv("NEO4J_USER"),
    password=os.getenv("NEO4J_PASSWORD"),
)
gc.verify_connectivity()

llm = ChatAnthropic(
    model="claude-sonnet-4-20250514",
    temperature=0.7,
    max_tokens=1024,
)

cypher_ret = CypherRetriever(gc, token_budget=2000)
vector_ret = VectorRetriever(persist_dir="./chroma_store_nb03")
extraction = ExtractionPipeline(llm=llm, graph_client=gc)

print("All components initialized.")
print(f"Graph state: {gc.get_node_counts()}")
print(f"ChromaDB collection count: {vector_ret._collection.count()}")

## 2. Tiered Cypher Retrieval — Individual Tiers

Let's examine each tier independently to verify they produce well-structured,
token-bounded context blocks.

In [ ]:
# T1: Active scene at the Citadel
t1 = cypher_ret.t1_active_scene("Citadel of Echoes", "main")
t1_tokens = count_tokens(t1)
print(f"=== T1: Active Scene ({t1_tokens} tokens) ===")
print(t1)
print()

In [ ]:
# T2: Causal chain from recent events
t2 = cypher_ret.t2_causal_chain(last_seq_id=3, branch_id="main")
t2_tokens = count_tokens(t2)
print(f"=== T2: Causal Chain ({t2_tokens} tokens) ===")
print(t2)
print()

# T3: Hostile tensions between present characters
present = ["Aria", "Kael"]
t3 = cypher_ret.t3_hostile_tensions(present, "main")
t3_tokens = count_tokens(t3) if t3 else 0
print(f"=== T3: Hostile Tensions ({t3_tokens} tokens) ===")
print(t3 if t3 else "(no hostile tensions found)")
print()

# T4: Orphan/dormant characters
t4 = cypher_ret.t4_orphan_nodes("main", current_seq_id=3)
t4_tokens = count_tokens(t4) if t4 else 0
print(f"=== T4: Orphan Hints ({t4_tokens} tokens) ===")
print(t4 if t4 else "(no orphan nodes found)")

In [ ]:
# Full tiered retrieval with budget enforcement
session = SessionState(
    session_id="nb03-demo",
    story_seed="The Citadel of Echoes",
    current_location="Citadel of Echoes",
    present_characters=["Aria", "Kael"],
    last_segment_seq_id=3,
    last_segment_text="Aria drew her sword as the shadows gathered.",
)

graph_context, graph_tokens = cypher_ret.retrieve(session)
print(f"=== Full Graph Context ({graph_tokens}/{cypher_ret._budget} tokens used) ===")
print(graph_context)
print(f"\nBudget utilization: {graph_tokens / cypher_ret._budget * 100:.1f}%")

## 3. Vector Retrieval — ChromaDB Tonal Anchors

Seed ChromaDB with existing segments from the graph, then query for
tonal anchors.

In [ ]:
# Seed ChromaDB with existing segments from the graph
seed_query = """
MATCH (s:Segment {branch_id: 'main'})
RETURN s.seq_id AS seq_id, s.text AS text
ORDER BY s.seq_id
"""
with gc._driver.session(database=gc._database) as db_session:
    segments = [(r["seq_id"], r["text"]) for r in db_session.run(seed_query)]

for seq_id, text in segments:
    vector_ret.add_segment(text, f"main_seq_{seq_id}")
    print(f"Indexed segment #{seq_id}: {text[:60]}...")

print(f"\nChromaDB collection size: {vector_ret._collection.count()} documents")

In [ ]:
# Retrieve tonal anchors
last_texts = [seg[1] for seg in segments[-3:]]
vector_context, vector_tokens = vector_ret.retrieve(last_texts, token_budget=1000)
print(f"=== Vector Context ({vector_tokens}/1000 tokens used) ===")
print(vector_context if vector_context else "(no tonal anchors)")
print(f"\nVector budget utilization: {vector_tokens / 1000 * 100:.1f}%")

## 4. Context Assembly

Merge graph facts + tonal anchors into the structured prompt sections
defined in Section 7.1 of the design document.

In [ ]:
assembled = ContextAssembler.assemble(
    graph_context=graph_context,
    vector_context=vector_context,
    violations=None,
    persona_docs=None,
    session=session,
)

for section, content in assembled.items():
    tokens = count_tokens(content)
    print(f"--- {section} ({tokens} tokens) ---")
    print(content[:200] + ("..." if len(content) > 200 else ""))
    print()

## 5. Full Pipeline — NKGE Mode

Run the complete LangGraph pipeline: retrieve → assemble → guard → generate → extract.

In [ ]:
pipeline = StoryPipeline(
    graph_client=gc,
    cypher_retriever=cypher_ret,
    vector_retriever=vector_ret,
    extraction=extraction,
    llm=llm,
    guard=None,  # Guard wired in P4
)

nkge_session = SessionState(
    session_id="nb03-nkge",
    story_seed="The Citadel of Echoes",
    current_location="Citadel of Echoes",
    present_characters=["Aria", "Kael"],
    last_segment_seq_id=3,
    last_segment_text="Aria drew her sword as the shadows gathered around the ancient doorway.",
    mode="nkge",
)

result_nkge = pipeline.run(nkge_session, player_action="Aria examines the glowing runes on the door")

print("=== NKGE Pipeline Result ===")
print(f"Graph context tokens: {result_nkge.get('graph_context_tokens', 0)}")
print(f"Vector context tokens: {result_nkge.get('vector_context_tokens', 0)}")
print(f"Violations: {len(result_nkge.get('violations', []))}")
print(f"\n--- Generated Segment ---")
print(result_nkge.get('generated_text', ''))
print(f"\n--- Extraction Result ---")
ext = result_nkge.get('extraction_result')
if ext:
    print(f"Proposals: {len(ext.proposals)}, Approved: {len(ext.approved)}, "
          f"Flagged: {len(ext.flagged)}, Committed: {ext.committed_count}")
    for p in ext.approved:
        print(f"  ✓ {p.entity_type}: {p.entity_name} (conf: {p.confidence})")

## 6. Baseline Mode Comparison

Run the same player action in baseline mode (rolling text summary,
no graph writes, no extraction). This produces the comparison point
for the P6 evaluation harness.

In [ ]:
baseline_session = SessionState(
    session_id="nb03-baseline",
    story_seed="The Citadel of Echoes",
    current_location="Citadel of Echoes",
    present_characters=["Aria", "Kael"],
    last_segment_seq_id=3,
    last_segment_text="Aria drew her sword as the shadows gathered around the ancient doorway.",
    mode="baseline",
)

result_baseline = pipeline.run(baseline_session, player_action="Aria examines the glowing runes on the door")

print("=== Baseline Pipeline Result ===")
print(f"Graph context tokens: {result_baseline.get('graph_context_tokens', 0)}")
print(f"Vector context tokens: {result_baseline.get('vector_context_tokens', 0)}")
print(f"Extraction result: {result_baseline.get('extraction_result')}")
print(f"\n--- Generated Segment (baseline) ---")
print(result_baseline.get('generated_text', ''))

## 7. Quantitative Comparison

Compare NKGE vs. Baseline across measurable dimensions.

In [ ]:
nkge_text = result_nkge.get('generated_text', '')
baseline_text = result_baseline.get('generated_text', '')

# Word counts
nkge_words = len(nkge_text.split())
baseline_words = len(baseline_text.split())

# Token counts
nkge_gen_tokens = count_tokens(nkge_text)
baseline_gen_tokens = count_tokens(baseline_text)

# Graph fact references (simple heuristic: count entity name mentions)
known_entities = gc.get_all_entity_names("main")
nkge_refs = sum(1 for e in known_entities if e.lower() in nkge_text.lower())
baseline_refs = sum(1 for e in known_entities if e.lower() in baseline_text.lower())

print("=== Quantitative Comparison ===")
print(f"{'Metric':<35} {'NKGE':>10} {'Baseline':>10}")
print("-" * 57)
print(f"{'Word count':<35} {nkge_words:>10} {baseline_words:>10}")
print(f"{'Token count':<35} {nkge_gen_tokens:>10} {baseline_gen_tokens:>10}")
print(f"{'Graph context tokens':<35} {result_nkge.get('graph_context_tokens', 0):>10} {result_baseline.get('graph_context_tokens', 0):>10}")
print(f"{'Vector context tokens':<35} {result_nkge.get('vector_context_tokens', 0):>10} {result_baseline.get('vector_context_tokens', 0):>10}")
print(f"{'Known entity references':<35} {nkge_refs:>10} {baseline_refs:>10}")
print(f"{'Entity reference rate':<35} {nkge_refs/max(len(known_entities),1)*100:>9.1f}% {baseline_refs/max(len(known_entities),1)*100:>9.1f}%")

ext = result_nkge.get('extraction_result')
if ext:
    print(f"{'Entities extracted (NKGE only)':<35} {len(ext.approved):>10} {'n/a':>10}")
    print(f"{'Entities committed (NKGE only)':<35} {ext.committed_count:>10} {'n/a':>10}")

## 8. Multi-Turn Pipeline Run

Run a 3-turn sequence to verify state accumulation: each turn should
see the previous turns' graph writes in its retrieval context.

In [ ]:
multi_session = SessionState(
    session_id="nb03-multi",
    story_seed="The Citadel of Echoes",
    current_location="Citadel of Echoes",
    present_characters=["Aria", "Kael"],
    last_segment_seq_id=3,
    last_segment_text="Aria drew her sword as the shadows gathered around the ancient doorway.",
    mode="nkge",
)

player_actions = [
    "Aria reaches out to touch the glowing runes carved into the ancient door",
    "Kael warns Aria about the danger and tries to pull her back",
    "The door opens, revealing a dark chamber beyond",
]

turn_metrics = []

for i, action in enumerate(player_actions):
    print(f"\n{'='*60}")
    print(f"TURN {i + 1}: {action}")
    print(f"{'='*60}")

    result = pipeline.run(multi_session, player_action=action)

    generated = result.get('generated_text', '')
    ext = result.get('extraction_result')

    # Update session for next turn
    multi_session.last_segment_seq_id += 1
    multi_session.last_segment_text = generated

    metrics = {
        "turn": i + 1,
        "graph_tokens": result.get('graph_context_tokens', 0),
        "vector_tokens": result.get('vector_context_tokens', 0),
        "word_count": len(generated.split()),
        "entities_extracted": len(ext.approved) if ext else 0,
        "entities_committed": ext.committed_count if ext else 0,
    }
    turn_metrics.append(metrics)

    print(f"\nGenerated ({metrics['word_count']} words):")
    print(generated[:300] + ("..." if len(generated) > 300 else ""))
    if ext:
        print(f"\nExtracted: {len(ext.approved)} entities, {ext.committed_count} committed")
        for p in ext.approved[:5]:
            print(f"  ✓ {p.entity_type}: {p.entity_name}")

print(f"\n\n{'='*60}")
print("MULTI-TURN METRICS SUMMARY")
print(f"{'='*60}")
print(f"{'Turn':<6} {'Graph Tok':<12} {'Vec Tok':<10} {'Words':<8} {'Extracted':<12} {'Committed':<10}")
print("-" * 58)
for m in turn_metrics:
    print(f"{m['turn']:<6} {m['graph_tokens']:<12} {m['vector_tokens']:<10} {m['word_count']:<8} {m['entities_extracted']:<12} {m['entities_committed']:<10}")

## 9. Graph State After Pipeline Runs

Verify the graph grew correctly from the extraction pipeline.

In [ ]:
node_counts = gc.get_node_counts()
rel_counts = gc.get_relationship_counts()

print("=== Graph State After Pipeline Runs ===")
print("\nNode counts:")
for label, count in sorted(node_counts.items()):
    print(f"  {label}: {count}")

print("\nRelationship counts:")
for rtype, count in sorted(rel_counts.items()):
    print(f"  {rtype}: {count}")

print(f"\nTotal nodes: {sum(node_counts.values())}")
print(f"Total relationships: {sum(rel_counts.values())}")

In [ ]:
# Clean up ChromaDB test collection
gc.close()
print("GraphClient closed. Notebook complete.")